In [0]:
import sys
sys.path.append("/Workspace/Users/yahyaghallali88@gmail.com/nyc_taxi-DB-DE/src")

In [0]:
from taxi_lakehouse.schema import apply_schema
from pyspark.sql import functions as F

In [0]:
base = "/Volumes/workspace/taxi_raw/landing"
table = "workspace.taxi_bronze.yellow_trips"
path = f"{base}/yellow/yellow_tripdata_2025-01_raw.parquet/"

raw_df = spark.read.parquet(path)
df = apply_schema(raw_df)

(
    df.write.format("delta")
    .mode("overwrite")
    .saveAsTable(table)
)

In [0]:
display(raw_df)

In [0]:
display(df)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

zone_schema = StructType([
    StructField("LocationID", IntegerType()),
    StructField("Borough", StringType()),
    StructField("Zone", StringType()),
    StructField("service_zone", StringType()),
])

zones = (spark.read.option("header", True).schema(zone_schema)
         .csv(f"{base}/zones/taxi_zone_lookup.csv"))

zones.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_bronze.taxi_zones")

display(zones.orderBy(F.col("LocationID").desc()).limit(10))